# 🎲 Module 2: Probability — Real-Life Cases

**The Story:** You're the data analyst at PizzaChain (50 stores). The CEO wants answers to questions like:
- "What's the chance a delivery is late?"
- "If a customer complained, what's the chance they'll churn?"
- "Our fraud detection flagged an order — is it really fraud?"

All of these are **probability** questions. Let's build the data and solve them step by step.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 2000  # 2000 orders

orders = pd.DataFrame({
    'order_id': range(1, n+1),
    'delivery_time_min': np.round(np.random.exponential(25, n) + 10, 1),
    'order_value': np.round(np.random.normal(22, 6, n), 2).clip(5),
    'is_weekend': np.random.choice([True, False], n, p=[0.35, 0.65]),
    'customer_complained': np.random.choice([True, False], n, p=[0.15, 0.85]),
})

# Late = delivery > 45 min
orders['is_late'] = orders['delivery_time_min'] > 45

# Churn: higher if complained
orders['churned'] = False
orders.loc[orders['customer_complained'], 'churned'] = np.random.choice(
    [True, False], orders['customer_complained'].sum(), p=[0.40, 0.60])
orders.loc[~orders['customer_complained'], 'churned'] = np.random.choice(
    [True, False], (~orders['customer_complained']).sum(), p=[0.08, 0.92])

print(f'Dataset: {len(orders)} orders')
orders.head(10)

---
## Case 1: Basic Probability — "What fraction of deliveries are late?"

**When to use:** You have a yes/no outcome and want to know how often it happens.

**Formula:** P(event) = count of event / total count

In [ ]:
total = len(orders)
late = orders['is_late'].sum()
p_late = late / total

print('📊 STEP-BY-STEP MATH')
print('=' * 50)
print(f'Total orders         = {total}')
print(f'Late orders (>45min) = {late}')
print(f'P(late) = {late} / {total} = {p_late:.4f} = {p_late*100:.1f}%')
print()
print(f'💡 INFERENCE: About {p_late*100:.0f}% of deliveries are late.')
print(f'   That means roughly 1 in {int(1/p_late)} orders arrives after 45 minutes.')
print(f'   If you get 200 orders/day, expect ~{int(200*p_late)} late deliveries daily.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pie chart
axes[0].pie([late, total-late], labels=[f'Late ({late})', f'On time ({total-late})'],
            colors=['#f45d6d', '#22d3a7'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[0].set_title('Delivery Status', fontsize=12, fontweight='bold')

# Histogram with cutoff
axes[1].hist(orders['delivery_time_min'], bins=30, color='#7c6aff', alpha=0.7, edgecolor='white')
axes[1].axvline(45, color='#f45d6d', linewidth=2, linestyle='--', label='Late threshold (45 min)')
axes[1].fill_betweenx([0, axes[1].get_ylim()[1] if axes[1].get_ylim()[1] > 0 else 100], 45, orders['delivery_time_min'].max(),
                       color='#f45d6d', alpha=0.15)
axes[1].set_xlabel('Delivery Time (min)')
axes[1].set_ylabel('Count')
axes[1].set_title('Delivery Time Distribution', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Case 2: Conditional Probability — "If a customer complained, what's the chance they churned?"

**When to use:** You want the probability of A **given that** B already happened.

**Formula:** P(A|B) = P(A and B) / P(B)

**Real-life question:** "Among customers who complained, what % left us?"

In [ ]:
complained = orders['customer_complained'].sum()
complained_and_churned = ((orders['customer_complained']) & (orders['churned'])).sum()
total_churned = orders['churned'].sum()
not_complained_churned = ((~orders['customer_complained']) & (orders['churned'])).sum()

p_churn_given_complaint = complained_and_churned / complained
p_churn_no_complaint = not_complained_churned / (total - complained)
p_churn_overall = total_churned / total

print('📊 STEP-BY-STEP MATH')
print('=' * 60)
print(f'Total orders              = {total}')
print(f'Complained                = {complained}')
print(f'Complained AND churned    = {complained_and_churned}')
print(f'Did NOT complain, churned = {not_complained_churned}')
print()
print(f'P(churn | complained)     = {complained_and_churned}/{complained} = {p_churn_given_complaint:.3f} = {p_churn_given_complaint*100:.1f}%')
print(f'P(churn | no complaint)   = {not_complained_churned}/{total-complained} = {p_churn_no_complaint:.3f} = {p_churn_no_complaint*100:.1f}%')
print(f'P(churn) overall          = {total_churned}/{total} = {p_churn_overall:.3f} = {p_churn_overall*100:.1f}%')
print()
print(f'💡 INFERENCE:')
print(f'   Customers who complained churn at {p_churn_given_complaint*100:.0f}% vs {p_churn_no_complaint*100:.0f}% for non-complainers.')
print(f'   That\'s {p_churn_given_complaint/p_churn_no_complaint:.1f}x higher! Complaints are a strong churn signal.')
print(f'   ACTION: Prioritize resolving complaints fast — it could cut churn significantly.')

In [ ]:
# Contingency table
ct = pd.crosstab(orders['customer_complained'], orders['churned'], margins=True)
ct.index = ['No Complaint', 'Complained', 'Total']
ct.columns = ['Stayed', 'Churned', 'Total']
print('📋 Contingency Table (the foundation of conditional probability):')
print(ct)

# Visual
fig, ax = plt.subplots(figsize=(8, 4))
x = ['Complained', 'No Complaint', 'Overall']
y = [p_churn_given_complaint*100, p_churn_no_complaint*100, p_churn_overall*100]
colors = ['#f45d6d', '#22d3a7', '#7c6aff']
bars = ax.bar(x, y, color=colors, alpha=0.8, edgecolor='white', width=0.5)
for bar, val in zip(bars, y):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%',
            ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate: Complained vs Not', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(y) + 10)
plt.tight_layout()
plt.show()

---
## Case 3: Bayes' Theorem — "Our fraud detector flagged an order. Is it really fraud?"

**When to use:** You have a test/detector result and want to know the TRUE probability of the condition.

**Formula:** P(fraud | flagged) = P(flagged | fraud) × P(fraud) / P(flagged)

**Why it matters:** Tests are never perfect. Even a 95% accurate fraud detector can be wrong most of the time if fraud is rare.

In [ ]:
# Setup: fraud is rare
p_fraud = 0.02          # 2% of orders are actually fraud
p_flag_if_fraud = 0.95  # detector catches 95% of real fraud
p_flag_if_legit = 0.04  # but falsely flags 4% of legit orders

# Bayes' step by step
p_flag = p_flag_if_fraud * p_fraud + p_flag_if_legit * (1 - p_fraud)
p_fraud_given_flag = (p_flag_if_fraud * p_fraud) / p_flag

print('📊 STEP-BY-STEP MATH (Bayes\' Theorem)')
print('=' * 60)
print(f'P(fraud)                = {p_fraud} (2% of orders are fraud)')
print(f'P(flagged | fraud)      = {p_flag_if_fraud} (detector catches 95% of fraud)')
print(f'P(flagged | legit)      = {p_flag_if_legit} (false alarm rate: 4%)')
print()
print(f'Step 1: P(flagged) = P(flag|fraud)×P(fraud) + P(flag|legit)×P(legit)')
print(f'       = {p_flag_if_fraud}×{p_fraud} + {p_flag_if_legit}×{1-p_fraud}')
print(f'       = {p_flag_if_fraud*p_fraud:.4f} + {p_flag_if_legit*(1-p_fraud):.4f}')
print(f'       = {p_flag:.4f}')
print()
print(f'Step 2: P(fraud | flagged) = P(flag|fraud)×P(fraud) / P(flagged)')
print(f'       = {p_flag_if_fraud*p_fraud:.4f} / {p_flag:.4f}')
print(f'       = {p_fraud_given_flag:.4f} = {p_fraud_given_flag*100:.1f}%')
print()
print(f'💡 INFERENCE:')
print(f'   Despite a 95% accurate detector, only {p_fraud_given_flag*100:.0f}% of flagged orders are real fraud!')
print(f'   Why? Because fraud is so rare (2%) that false alarms overwhelm true catches.')
print(f'   ACTION: Don\'t auto-block flagged orders. Add a second verification step.')

In [ ]:
# Visual: out of 10,000 orders
N = 10000
real_fraud = int(N * p_fraud)
legit = N - real_fraud
true_pos = int(real_fraud * p_flag_if_fraud)
false_pos = int(legit * p_flag_if_legit)
total_flagged = true_pos + false_pos

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the 10,000 orders breakdown
categories = ['Real Fraud\n(caught)', 'Real Fraud\n(missed)', 'Legit\n(falsely flagged)', 'Legit\n(correct)']
values = [true_pos, real_fraud - true_pos, false_pos, legit - false_pos]
colors = ['#f45d6d', '#f5b731', '#e879a8', '#22d3a7']
bars = axes[0].bar(categories, values, color=colors, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title(f'Out of {N:,} Orders', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')

# Right: of those flagged, how many are real?
axes[1].pie([true_pos, false_pos],
            labels=[f'Real Fraud ({true_pos})', f'False Alarm ({false_pos})'],
            colors=['#f45d6d', '#e879a8'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title(f'Of {total_flagged} Flagged Orders, How Many Are Real?', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'💡 {false_pos} innocent orders get flagged for every {true_pos} real fraud catches.')
print(f'   This is why Bayes\' Theorem matters — raw accuracy is misleading when the event is rare.')

---
## Case 4: Law of Large Numbers — "Why do we need more data?"

**When to use:** To understand why small samples are unreliable and large samples converge to the truth.

In [ ]:
# Simulate: track the running late-delivery rate as we see more orders
np.random.seed(42)
is_late_seq = (np.random.exponential(25, 5000) + 10) > 45
running_rate = np.cumsum(is_late_seq) / np.arange(1, 5001)
true_rate = is_late_seq.mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(running_rate, color='#7c6aff', linewidth=1.5, label='Running estimate')
ax.axhline(true_rate, color='#22d3a7', linewidth=2, linestyle='--', label=f'True rate: {true_rate:.3f}')
ax.fill_between(range(5000), true_rate - 0.02, true_rate + 0.02, color='#22d3a7', alpha=0.1)
ax.set_xlabel('Number of Orders Observed')
ax.set_ylabel('Estimated Late Rate')
ax.set_title('Law of Large Numbers: More Data → Better Estimate', fontsize=12, fontweight='bold')
ax.legend()

# Annotations
ax.annotate('Wild swings\n(small sample)', xy=(50, running_rate[50]), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#f45d6d'), xytext=(300, running_rate[50]+0.08), color='#f45d6d')
ax.annotate('Stable estimate\n(large sample)', xy=(4000, running_rate[4000]), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#22d3a7'), xytext=(3500, running_rate[4000]+0.06), color='#22d3a7')

plt.tight_layout()
plt.show()

print(f'💡 INFERENCE:')
print(f'   After 10 orders, the estimate could be anywhere from 0% to 40%.')
print(f'   After 5000 orders, it settles near the true rate of {true_rate*100:.1f}%.')
print(f'   LESSON: Never make decisions from small samples. Wait for enough data.')

---
## 📝 Summary

| Case | Method | Question | Key Insight |
|---|---|---|---|
| Late deliveries | Basic probability | What fraction are late? | ~1 in N orders is late |
| Complaint → Churn | Conditional probability | If complained, will they leave? | Complainers churn Nx more |
| Fraud detection | Bayes' Theorem | Flagged = really fraud? | Rare events → most flags are false alarms |
| Sample size | Law of Large Numbers | How much data do we need? | Small samples lie, large samples converge |